HW2 pilot: p_para vs p_infer, 20 assets, pilot V4: new p_infer fragment, same 20 assets, no metrics. Do not rerun after seeing outputs.

In [ ]:
!pip -q install "transformers==4.57.6"

In [ ]:
import sys, torch, transformers
print('python', sys.version.split()[0])
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('compute_capability', torch.cuda.get_device_capability(0))

In [ ]:
from google.colab import files
uploaded = files.upload()
print(sorted(uploaded))

In [ ]:
from pathlib import Path
import shutil
base = Path('/content')
(base/'my_llm').mkdir(exist_ok=True)
(base/'FAR-Trans').mkdir(exist_ok=True)
(base/'starter'/'far-trans').mkdir(parents=True, exist_ok=True)
for name in ['encoder.py', 'data_adapter.py', 'runtime.py', 'pilot_assets.json']:
    shutil.move(str(base/name), str(base/'my_llm'/name))
for name in ['transactions.csv', 'asset_information.csv',
             'customer_information.csv', 'close_prices.csv']:
    shutil.move(str(base/name), str(base/'FAR-Trans'/name))
shutil.move(str(base/'starter.py'), str(base/'starter'/'far-trans'/'starter.py'))

In [ ]:
!python /content/my_llm/encoder.py \
  --pilot-assets /content/my_llm/pilot_assets.json \
  --data-dir /content/FAR-Trans \
  --starter-dir /content/starter/far-trans \
  --output-dir /content/pilot_v4 \
  --model Qwen/Qwen3-4B-Instruct-2507 \
  --revision cdbee75f17c01a7cc42f958dc650907174af0554 \
  --device cuda --allow-download

In [ ]:
import json
from collections import Counter
from pathlib import Path
rows = [json.loads(line) for line in
        Path('/content/pilot_v4/generations.jsonl').open(encoding='utf-8')]
counts = Counter(r['prompt_mode'] for r in rows)
print('records', len(rows), 'assets', len({r['asset_id'] for r in rows}))
print('modes', dict(sorted(counts.items())))
assert len(rows) == 60
assert len({r['asset_id'] for r in rows}) == 20
assert counts == {'para': 20, 'noecho': 20, 'infer': 20}
print(Path('/content/pilot_v4/pilot_manifest.json').read_text(encoding='utf-8'))

In [ ]:
from google.colab import files
files.download('/content/pilot_v4/generations.jsonl')
files.download('/content/pilot_v4/pilot_manifest.json')